### "Statistics and Data Analysis" 28: Bayesian Inference: Overview
In this lecture, Dr. Brunton introduces Bayesian inference and statistics, which is a powerful framework for learning distributions from data. 

https://www.youtube.com/watch?v=XCEpIBqKogo&list=PLMrJAkhIeNNT14qn1c5qdL29A1UaHamjx&index=28

In [ ]:
# Simple example where we have two hypotheses

import numpy as np

# Prior probabilities for fair (H1) and biased (H2) coin
p_h1 = 0.5 # prior probability of Fair coin (H1)
p_h2 = 0.5 # prior probability of Biased coin (H2)

# Likelihood of the data (coin flip result) under each hypothesis
def likelihood(data, hypothesis):
    if hypothesis == 'H1':  # Fair coin
        return 0.5
    elif hypothesis == 'H2':  # Biased coin (70% heads)
        return 0.7 if data == 'H' else 0.3

# Update beliefs using Bayes' Theorem
def bayesian_update(prior_h1, prior_h2, data):
    # Likelihoods: P(data | hypothesis)
    likelihood_h1 = likelihood(data, 'H1')
    likelihood_h2 = likelihood(data, 'H2')
        
    # Total evidence P(data): no other hypotheses considered
    p_data = (likelihood_h1 * prior_h1) + (likelihood_h2 * prior_h2)

    # Posteriors: P(hypothesis | data)
    posterior_h1 = (likelihood_h1 * prior_h1) / p_data
    posterior_h2 = (likelihood_h2 * prior_h2) / p_data
    
    return posterior_h1, posterior_h2

# Simulate a series of coin flips 
# (e.g., a Fair coin with p=[0.5, 0.5] or a Biased coin with p=[0.7, 0.3])
coin_flips = np.random.choice(['H', 'T'], size=20, p=[0.5, 0.5])

p_h1_posterior, p_h2_posterior = p_h1, p_h2
for flip in coin_flips:
    p_h1_posterior, p_h2_posterior = bayesian_update(p_h1_posterior, p_h2_posterior, flip)
    print(f"After observing {flip}: P(H1|data) = {p_h1_posterior:.4f}, P(H2|data) = {p_h2_posterior:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta

# Function to compute the likelihood of the data given theta
def likelihood(theta, data):
    # Binomial likelihood for coin flips
    heads = np.sum(data=='H')
    tails = np.sum(data=='T')
    return theta**heads * (1-theta)**tails

# Function to compute the posterior distribution using Bayesian updating
def bayesian_update(prior_alpha, prior_beta, data):
    # calculate the number of heads and tails in the data
    heads = np.sum(data=='H')
    tails = np.sum(data=='T')

    # Update the prior (beta distribution) parameters
    posterior_alpha = prior_alpha + heads
    posterior_beta = prior_beta + tails

    return posterior_alpha, posterior_beta

# Start with a Prior: Uniform distribution (Beta(1, 1))
prior_alpha = 1
prior_beta = 1

# Simulate a sequence of coin flips (H for heads, T for tails)
coin_flips = np.array(['H','T','H','H','T','H','T','T'])
# coin_flips = np.random.choice(['H', 'T'], size=10, p=[0.5, 0.5])

# Store prior values for plotting
alphas = [prior_alpha]
betas = [prior_beta]

# Perform Bayesian updates after each coin flip
for i, flip in enumerate(coin_flips):
    print(f"Before Bayesian update: prior_alpha={prior_alpha}, prior_beta={prior_beta}, flip={flip}")
    
    # Update the posterior with the new data (one flip)
    prior_alpha, prior_beta = bayesian_update(prior_alpha, prior_beta, np.array([flip]))
    
    # Store the updated posterior parameters
    alphas.append(prior_alpha)
    betas.append(prior_beta)

    # Print the updated posterior parameters after each coin flip
    print(f"After observing {i+1} flips ({flip}: Posterior alpha={prior_alpha}, beta={prior_beta})")

# Plot the evolution of the posterior distribution after all flips
theta_range = np.linspace(0, 1, 100)
plt.figure(figsize=(10, 6))

cmap = plt.get_cmap('viridis_r')
colors = cmap(np.linspace(0, 1, len(alphas)))

for i in range(len(alphas)):
    posterior_pdf = beta.pdf(theta_range, alphas[i], betas[i])
    plt.plot(theta_range, posterior_pdf, color=colors[i], label=f"After {i} flips")

# Customize the plot
plt.title("Evolution of Posterior Distribution with Each Coin Flip")
plt.xlabel("Theta (probability of heads)")
plt.ylabel("Density")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Generate 100 random coin flips (1 for heads, 0 for tails)
coin_flips = np.random.choice([1, 0], size=100)

# Initial prior: Beta(1, 1) distribution (uniform prior)
alpha_prior = 1
beta_prior = 1

# Set up the figure with two subplots: one for the density and one for the mean
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8))

# Adjust the space between the subplots
plt.subplots_adjust(hspace=0.3)

# Set up the first plot for Beta distribution
theta_range = np.linspace(0, 1, 1000)
line1, = ax1.plot([], [], lw=2)
mean_line, = ax1.plot([], [], lw=2, linestyle='dashed', color='orange')  # Dashed line for mean
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 5)  # Set initial y-limit, we'll update this dynamically later
ax1.set_title('Beta Distribution Evolution')
ax1.set_xlabel('Theta (probability of heads)')
ax1.set_ylabel('Density')

# Set up the second plot for mean evolution
line2, = ax2.plot([], [], lw=2, color='orange')
ax2.set_xlim(0, len(coin_flips))
ax2.set_ylim(0, 1)
ax2.set_title('Evolution of Posterior Mean')
ax2.set_xlabel('Flip Number')
ax2.set_ylabel('Posterior Mean')

# Initialize shaded regions for both plots
std_shade1 = ax1.fill_between([], [], [], color='orange', alpha=0.2)
std_shade2 = ax2.fill_between([], [], [], color='orange', alpha=0.2)

# To store the evolution of the mean and std after each flip
posterior_means = [alpha_prior / (alpha_prior + beta_prior)]  # Start with the prior mean
posterior_stds = [np.sqrt((alpha_prior * beta_prior) / ((alpha_prior + beta_prior)**2 * (alpha_prior + beta_prior + 1)))]  # Start with prior std

# Function to initialize the plots
def init():
    # Plot the initial prior distribution (Beta(1, 1))
    density = beta.pdf(theta_range, alpha_prior, beta_prior)
    line1.set_data(theta_range, density)
    mean_line.set_data([], [])
    return line1, mean_line, line2

# Function to update the plots after each coin flip
def update(frame):
    global alpha_prior, beta_prior, std_shade1, std_shade2
    
    # If frame == 0, we are showing the prior, so don't update
    if frame > 0:
        flip = coin_flips[frame - 1]  # Adjust to reflect that the first frame is the prior
        
        # Update the Beta distribution parameters
        alpha_prior += flip  # Add to alpha for heads (1)
        beta_prior += (1 - flip)  # Add to beta for tails (0)
        
    # Calculate the posterior mean and standard deviation for Beta(alpha, beta)
    posterior_mean = alpha_prior / (alpha_prior + beta_prior)
    posterior_std = np.sqrt((alpha_prior * beta_prior) / ((alpha_prior + beta_prior)**2 * (alpha_prior + beta_prior + 1)))
    
    if frame > 0:  # Only append if it's a flip
        posterior_means.append(posterior_mean)
        posterior_stds.append(posterior_std)

    # Update the line for the Beta distribution (density plot)
    density = beta.pdf(theta_range, alpha_prior, beta_prior)
    line1.set_data(theta_range, density)

    # Dynamically update the y-axis limit based on the max value of the density
    ax1.set_ylim(0, 1.1 * np.max(density))
    
    # Update the line for the evolution of the mean
    line2.set_data(np.arange(len(posterior_means)), posterior_means)

    # Remove the previous shaded region (properly clearing it)
    for coll in ax1.collections:
        coll.remove()
    for coll in ax2.collections:
        coll.remove()

    # Update the shaded region for the Beta distribution plot (±1 std deviation)
    std_shade1 = ax1.fill_between(
        theta_range,
        beta.pdf(theta_range, alpha_prior, beta_prior) * (theta_range >= posterior_mean - posterior_std) * (theta_range <= posterior_mean + posterior_std),
        color='orange',
        alpha=0.4
    )

    # Update the shaded region for the mean plot (±1 std deviation)
    std_shade2 = ax2.fill_between(
        np.arange(len(posterior_means)),
        np.array(posterior_means) - np.array(posterior_stds),
        np.array(posterior_means) + np.array(posterior_stds),
        color='orange',
        alpha=0.4
    )
    
    # Update the vertical dashed line for the mean in the Beta plot
    mean_line.set_data([posterior_mean, posterior_mean], [0, np.max(density)]) # Dashed line from y=0 to max density
    
    # Update titles
    ax1.set_title(f'After flip {frame}: {"Prior" if frame == 0 else ("Heads" if flip == 1 else "Tails")}')
    
    return line1, mean_line, line2

# Create the animation, including the initial prior frame
ani = FuncAnimation(fig, update, frames=len(coin_flips) + 1, init_func=init, blit=True, repeat=False)

# Display the animation in Jupyter Notebook using HTML
HTML(ani.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Generate 100 random coin flips (1 for heads, 0 for tails)
coin_flips = np.random.choice([1, 0], size=1000)

# Initial prior: Beta(1, 1) distribution (uniform prior)
alpha_prior = 1
beta_prior = 1

# Set up the figure with two subplots: one for the density and one for the mean
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8))

# Adjust the space between the subplots
plt.subplots_adjust(hspace=0.3)

# Set up the first plot for Beta distribution
theta_range = np.linspace(0, 1, 1000)
line1, = ax1.plot([], [], lw=2)
mean_line, = ax1.plot([], [], lw=2, linestyle='dashed', color='orange')  # Dashed line for mean
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 5)  # Set initial y-limit, we'll update this dynamically later
ax1.set_title('Beta Distribution Evolution')
ax1.set_xlabel('Theta (probability of heads)')
ax1.set_ylabel('Density')

# Set up the second plot for mean evolution
line2, = ax2.plot([], [], lw=2, color='orange')
ax2.set_xlim(0, len(coin_flips))
ax2.set_ylim(0, 1)
ax2.set_title('Evolution of Posterior Mean')
ax2.set_xlabel('Flip Number')
ax2.set_ylabel('Posterior Mean')

# Initialize shaded regions for both plots
std_shade1 = ax1.fill_between([], [], [], color='orange', alpha=0.2)
std_shade2 = ax2.fill_between([], [], [], color='orange', alpha=0.2)

# To store the evolution of the mean and std after each flip
posterior_means = [alpha_prior / (alpha_prior + beta_prior)]  # Start with the prior mean
posterior_stds = [np.sqrt((alpha_prior * beta_prior) / ((alpha_prior + beta_prior)**2 * (alpha_prior + beta_prior + 1)))]  # Start with prior std

# Function to initialize the plots
def init():
    # Plot the initial prior distribution (Beta(1, 1))
    density = beta.pdf(theta_range, alpha_prior, beta_prior)
    line1.set_data(theta_range, density)
    mean_line.set_data([], [])
    return line1, mean_line, line2

# Function to update the plots after every 10 coin flips
def update(frame):
    global alpha_prior, beta_prior, std_shade1, std_shade2
    
    # If frame == 0, we are showing the prior, so don't update
    if frame > 0:
        # Accumulate 10 flips at a time
        flips = coin_flips[(frame - 1)*10: frame * 10]  # Adjust to reflect that the first frame is the prior
        heads = np.sum(flips) # Number of heads in the last 10 flips
        tails = len(flips) - heads # Number of tails in the last 10 flips

        # Update the Beta distribution parameters
        alpha_prior += heads  # Add to alpha for heads (1)
        beta_prior += tails  # Add to beta for tails (0)
        
    # Calculate the posterior mean and standard deviation for Beta(alpha, beta)
    posterior_mean = alpha_prior / (alpha_prior + beta_prior)
    posterior_std = np.sqrt((alpha_prior * beta_prior) / ((alpha_prior + beta_prior)**2 * (alpha_prior + beta_prior + 1)))
    
    if frame > 0:  # Only append if it's a flip
        posterior_means.append(posterior_mean)
        posterior_stds.append(posterior_std)

    # Update the line for the Beta distribution (density plot)
    density = beta.pdf(theta_range, alpha_prior, beta_prior)
    line1.set_data(theta_range, density)

    # Dynamically update the y-axis limit based on the max value of the density
    ax1.set_ylim(0, 1.1 * np.max(density))
    
    # Update the line for the evolution of the mean
    line2.set_data(np.arange(len(posterior_means))*10, posterior_means) # Scale by 10 flips

    # Remove the previous shaded region (properly clearing it)
    for coll in ax1.collections:
        coll.remove()
    for coll in ax2.collections:
        coll.remove()

    # Update the shaded region for the Beta distribution plot (±1 std deviation)
    std_shade1 = ax1.fill_between(
        theta_range,
        beta.pdf(theta_range, alpha_prior, beta_prior) * (theta_range >= posterior_mean - posterior_std) * (theta_range <= posterior_mean + posterior_std),
        color='orange',
        alpha=0.4
    )

    # Update the shaded region for the mean plot (±1 std deviation)
    std_shade2 = ax2.fill_between(
        np.arange(len(posterior_means)) * 10,
        np.array(posterior_means) - np.array(posterior_stds),
        np.array(posterior_means) + np.array(posterior_stds),
        color='orange',
        alpha=0.4
    )
    
    # Update the vertical dashed line for the mean in the Beta plot
    mean_line.set_data([posterior_mean, posterior_mean], [0, np.max(density)]) # Dashed line from y=0 to max density
    
    # Update titles
    if frame ==0:
        ax1.set_title('Prior Distribution (Beta(1, 1))')
    else:
        ax1.set_title(f'After flip {frame * 10}: Heads = {heads}, Tails={tails}')
    
    return line1, mean_line, line2

# Create the animation, including the initial prior frame and updating every 10 flips
frames = len(coin_flips) // 10 + 1  # +1 for the prior frame
ani = FuncAnimation(fig, update, frames=frames, init_func=init, blit=True, repeat=False)

# Display the animation in Jupyter Notebook using HTML
HTML(ani.to_jshtml())